# Exkurs: Eine eigene Sentiment-Analyse selbst trainieren

**Optionale Vertiefung — kein Pflichtstoff**

> ℹ️ **Hinweis:** Dieses Notebook ist ein **Exkurs**. Du brauchst es nicht, um den Kern des
> Kapitels zu verstehen — es lohnt sich aber, wenn Du wissen willst, was hinter den Kulissen
> eines "eigenen" NLP-Modells steckt, statt nur ein fertiges zu laden.

## Real-World-Kontext

In der anderen Übung dieses Kapitels hast Du ein **vortrainiertes** HuggingFace-Modell
(`nlptown/bert-base-multilingual-uncased-sentiment`) geladen und sofort für Sentiment-Vorhersagen
auf Kfz-Versicherer-Kundenbewertungen genutzt. Kein Training, keine eigenen Daten — nur ein paar
Zeilen Code, und das Modell lieferte Ergebnisse.

Aber was, wenn Du kein Internet, keine Cloud-API und kein passendes vortrainiertes Modell zur
Verfügung hast? Was, wenn Deine Sprache oder Dein Fachvokabular so speziell ist, dass kein
fertiges Modell existiert? Dann bleibt nur eine Option: **Du trainierst selbst.**

In diesem Notebook baust Du genau das — von Grund auf, mit PyTorch, auf einem eigenen kleinen
Datensatz aus 350 synthetischen Kfz-Versicherer-Kundenbewertungen. Am Ende vergleichst Du ehrlich,
wie viel schlechter (oder besser) Dein selbst trainiertes Modell im Vergleich zum vortrainierten
BERT-Modell abschneidet — und verstehst, **warum**.

## 🎯 Learning Objectives

By completing this notebook, you will be able to:
- 💡 **Explain** warum man ein NLP-Modell manchmal selbst trainieren muss, statt ein
  vortrainiertes Modell zu laden, und welche Kompromisse das mit sich bringt.
- 🔍 **Identify** die drei Bausteine einer minimalen Sentiment-Architektur: **Embedding**,
  **Global Average Pooling** und **Klassifikationskopf**.
- 🛠️ **Apply** einen eigenen Trainings-Pipeline-Schritt (Tokenisierung, Vokabular-Aufbau,
  Padding) auf einen deutschsprachigen Text-Datensatz an.
- 🛠️ **Apply** `PyTorch` (`nn.Embedding`, `nn.Linear`), um einen eigenen
  `SelfTrainedSentimentClassifier` zu definieren, zu trainieren und zu evaluieren.
- ⚖️ **Bewerten**, wann sich Self-Training gegenüber einem vortrainierten Modell trotz
  schlechterer Accuracy lohnt.

## Concept at a Glance

**Analogie — ein Modell wie ein Kind, das gerade erst Wörter lernt:** Ein vortrainiertes Modell
wie BERT ist wie ein Erwachsener, der schon Millionen Bücher, Zeitungen und Gespräche gehört hat —
er kennt Nuancen, Ironie, seltene Wörter. Unser selbst trainiertes Modell dagegen ist wie ein Kind,
dem Du an einem einzigen Nachmittag genau **245 Sätze** vorliest und sagst: "Das hier war positiv
gemeint, das hier negativ." Das Kind wird danach schon ein Gefühl für ein paar Signalwörter haben
("super", "schnell", "unfreundlich") — aber es wird bei weitem nicht so differenziert urteilen
können wie der Erwachsene.

**Die Architektur, die wir bauen** (analog zum `VersichererMLP` aus dem letzten Kapitel, aber mit
Text statt Zahlen als Input):

1. **Embedding-Schicht:** Jedes Wort wird in einen Vektor aus Zahlen übersetzt (ähnliche Wörter
   sollen später ähnliche Vektoren bekommen).
2. **Global Average Pooling:** Alle Wortvektoren eines Satzes werden **gemittelt** zu einem
   einzigen Satzvektor — egal wie lang der Satz ist, am Ende steht immer ein Vektor fester Länge.
3. **Klassifikationskopf:** Eine kleine Fully-Connected-Schicht (`nn.Linear` + `ReLU` +
   `nn.Linear`) wandelt den Satzvektor in drei Ausgänge um — je einen für **Positiv**, **Neutral**
   und **Negativ**.

> 💡 **Good to know:**
> Genau dieses Design wurde auf der Vorlesungsfolie "Modell-Architektur für Self-Trained
> Sentiment" gezeigt: Embedding → Global Average Pooling → MLP-Head → Softmax. Es ist bewusst
> **minimal** gehalten — einfach genug, um mit nur 245 Trainingsbeispielen überhaupt trainierbar
> zu sein.

Unser Running Example bleibt derselbe Kfz-Versicherer-Case wie im gesamten Kurs — nur diesmal
sind die Bewertungen echter, unstrukturierter **Text** statt Zahlen in einer Tabelle.

## Schritt 1 — Synthetischer Text-Datensatz: 350 Kundenbewertungen

Weil wir offline arbeiten und reproduzierbare Ergebnisse brauchen, generieren wir unsere
Kundenbewertungen synthetisch — mit demselben Template-Ansatz wie schon in früheren Kapiteln
(`numpy.random.default_rng` mit festem Seed statt eines externen Downloads).

Wir bauen die Bewertungen aus austauschbaren **Satzbausteinen** zu vier Aspekten zusammen, die bei
einer Kfz-Versicherung typischerweise bewertet werden:

- **Wartezeit** bei der Schadensbearbeitung
- **Erreichbarkeit** der Hotline
- **Erstattung** der Schadenshöhe
- **Freundlichkeit** des Sachbearbeiters

Für jeden Aspekt gibt es passende Satzbausteine in drei Tonalitäten: positiv, neutral, negativ.

In [ ]:
# I DO: Satzbausteine je Aspekt und Tonalitaet definieren

bausteine = {
    "wartezeit": {
        "positiv": [
            "Die Bearbeitung meines Schadens ging erstaunlich schnell.",
            "Ich hatte meine Rückmeldung bereits nach zwei Tagen.",
            "Alles wurde zügig und ohne Verzögerung abgewickelt.",
        ],
        "neutral": [
            "Die Bearbeitung hat ein paar Tage gedauert, im üblichen Rahmen.",
            "Ich musste etwa eine Woche auf eine Antwort warten.",
            "Die Bearbeitungszeit war weder besonders schnell noch besonders langsam.",
        ],
        "negativ": [
            "Ich musste wochenlang auf eine Rückmeldung warten.",
            "Die Bearbeitung hat sich unnötig in die Länge gezogen.",
            "Es hat viel zu lange gedauert, bis sich überhaupt jemand gemeldet hat.",
        ],
    },
    "erreichbarkeit": {
        "positiv": [
            "Die Hotline war sofort erreichbar, ich kam ohne Wartezeit durch.",
            "Ich wurde beim ersten Anruf direkt mit der richtigen Person verbunden.",
            "Die telefonische Erreichbarkeit war vorbildlich.",
        ],
        "neutral": [
            "Die Erreichbarkeit war in Ordnung, ich musste aber zweimal anrufen.",
            "Ich kam nach kurzer Warteschleife durch, das war akzeptabel.",
            "Telefonisch war man erreichbar, aber nicht sonderlich schnell.",
        ],
        "negativ": [
            "Telefonisch war ich tagelang nicht durchgekommen.",
            "Die Hotline war ständig überlastet und schwer erreichbar.",
            "Niemand hat meine Anrufe zurückgerufen.",
        ],
    },
    "erstattung": {
        "positiv": [
            "Die Schadenssumme wurde vollständig und fair erstattet.",
            "Ich habe sogar mehr zurückbekommen, als ich erwartet hatte.",
            "Die Erstattung entsprach genau meiner Erwartung.",
        ],
        "neutral": [
            "Die Erstattung kam wie erwartet, weder mehr noch weniger.",
            "Der Schaden wurde in etwa nach Standard reguliert.",
            "Die Höhe der Erstattung war nachvollziehbar, aber nicht überragend.",
        ],
        "negativ": [
            "Die Erstattung war deutlich niedriger als erwartet.",
            "Der Schaden wurde nur teilweise anerkannt, ohne nachvollziehbare Begründung.",
            "Ich habe am Ende viel weniger Geld erhalten, als mir zustand.",
        ],
    },
    "freundlichkeit": {
        "positiv": [
            "Der Sachbearbeiter war sehr freundlich und hat mir alles genau erklärt.",
            "Ich wurde die ganze Zeit über kompetent und herzlich betreut.",
            "Der Kundenservice war ausgesprochen zuvorkommend.",
        ],
        "neutral": [
            "Der Sachbearbeiter war sachlich, aber nicht besonders herzlich.",
            "Der Kontakt verlief professionell, aber unpersönlich.",
            "Man wurde ordentlich behandelt, mehr aber auch nicht.",
        ],
        "negativ": [
            "Der Sachbearbeiter war unfreundlich und wenig hilfsbereit.",
            "Ich fühlte mich mit meinen Fragen komplett alleingelassen.",
            "Der Ton am Telefon war teilweise regelrecht schroff.",
        ],
    },
}

print("Aspekte:", list(bausteine.keys()))
print("Beispiel (Erstattung, negativ):", bausteine["erstattung"]["negativ"][0])

### Bewertungen aus Bausteinen zusammensetzen

Jetzt bauen wir eine Funktion, die pro Bewertung **eine feste Tonalität** (das spätere Label)
festlegt und dann 2–3 zufällig ausgewählte Aspekte in genau dieser Tonalität zu einem Fließtext
zusammensetzt. So entstehen viele unterschiedliche, aber inhaltlich konsistente Bewertungen aus
denselben Bausteinen.

In [ ]:
# I DO: Funktion zum Zusammensetzen einer einzelnen Bewertung

import numpy as np

def erstelle_bewertung(tonalitaet, rng):
    # Waehle 2 oder 3 der 4 Aspekte zufaellig aus (ohne Wiederholung)
    anzahl_aspekte = rng.integers(2, 4)  # 2 oder 3
    aspekte = list(bausteine.keys())
    gewaehlte_aspekte = rng.choice(aspekte, size=anzahl_aspekte, replace=False)

    # Fuer jeden gewaehlten Aspekt einen passenden Satz in der gewuenschten Tonalitaet ziehen
    saetze = [
        rng.choice(bausteine[aspekt][tonalitaet])
        for aspekt in gewaehlte_aspekte
    ]
    return " ".join(saetze)

# Kurzer Test mit einem festen Seed
test_rng = np.random.default_rng(0)
print(erstelle_bewertung("positiv", test_rng))
print(erstelle_bewertung("negativ", test_rng))

> 💡 **Good to know:**
> Wir ziehen die Aspekte **ohne Wiederholung** (`replace=False`), damit nicht zweimal derselbe
> Aspekt (z. B. zweimal etwas zur Wartezeit) in einer Bewertung landet. Innerhalb eines Aspekts
> ziehen wir dagegen **mit Wiederholung** — dieselbe Satzformulierung darf über verschiedene
> Bewertungen hinweg mehrfach vorkommen, das ist bei echten Kundenbewertungen genauso.

Jetzt erzeugen wir alle 350 Bewertungen: möglichst gleichmäßig über die drei Klassen verteilt.

In [ ]:
# I DO: 350 Bewertungen generieren, moeglichst gleichmaessig ueber 3 Klassen verteilt

import pandas as pd

rng = np.random.default_rng(42)  # fester Seed -> reproduzierbare Ergebnisse
n_gesamt = 350

# Anzahl Bewertungen pro Klasse: moeglichst gleich (117 / 117 / 116)
anzahl_pro_klasse = [117, 117, 116]
tonalitaeten = ["positiv", "neutral", "negativ"]

bewertungen, wahre_tonalitaeten = [], []
for tonalitaet, anzahl in zip(tonalitaeten, anzahl_pro_klasse):
    for _ in range(anzahl):
        bewertungen.append(erstelle_bewertung(tonalitaet, rng))
        wahre_tonalitaeten.append(tonalitaet)

kundenbewertungen = pd.DataFrame({"text": bewertungen, "wahre_tonalitaet": wahre_tonalitaeten})
print(f"Anzahl Bewertungen: {len(kundenbewertungen)}")
kundenbewertungen.head()

### Realistische Unschärfe: Label-Rauschen

Echte Kundenbewertungen werden nicht von einer Maschine, sondern von **Menschen** annotiert (oder
über Sternebewertungen abgeleitet) — und Menschen sind sich nicht immer einig, ob eine Bewertung
"eher neutral" oder "schon leicht negativ" ist. Ein Text mit gemischten Signalen wird von zwei
verschiedenen Personen manchmal unterschiedlich eingeordnet.

Damit unser synthetischer Datensatz nicht *unrealistisch sauber* ist (bei rein template-basierten
Sätzen wäre er sonst nahezu perfekt nach Stichwörtern trennbar — echte Sprache ist nie so
eindeutig), simulieren wir dieses **Label-Rauschen** explizit: Bei einem Teil der Bewertungen
weicht das tatsächlich vergebene Label bewusst von der "wahren" Tonalität ab, mit der der Text
gebaut wurde.

> ⚠️ **Common Pitfall:**
> Das ist **kein Trick**, um am Ende auf eine bestimmte Ziel-Accuracy zu kommen — es ist eine
> ehrliche Modellierung eines Effekts, den es in jedem realen, von Menschen gelabelten Datensatz
> gibt. Wir legen die Rauschstärke (`RAUSCHANTEIL`) einmal fest, *bevor* wir irgendetwas trainieren
> — welche Test-Accuracy am Ende dabei herauskommt, sehen wir erst später.
>
> Ohne dieses Rauschen wären unsere Vorlage-Sätze rein anhand einzelner Signalwörter (z. B.
> "freundlich" vs. "unfreundlich") nahezu perfekt trennbar — echte Kundenbewertungen sind nie so
> eindeutig. Der gewählte Wert von 25 % ist eine bewusste didaktische Entscheidung, damit die
> Aufgabe für unser einfaches Modell wieder eine echte Herausforderung bleibt.

In [ ]:
# I DO: Label-Rauschen simulieren - realistische Uneindeutigkeit der Annotation

RAUSCHANTEIL = 0.25  # 25% der Bewertungen bekommen ein abweichendes Label

label_rng = np.random.default_rng(7)  # eigener, fester Seed fuer das Rauschen
finale_labels = []
for wahre_tonalitaet in kundenbewertungen["wahre_tonalitaet"]:
    if label_rng.random() < RAUSCHANTEIL:
        andere_klassen = [t for t in tonalitaeten if t != wahre_tonalitaet]
        finale_labels.append(label_rng.choice(andere_klassen))
    else:
        finale_labels.append(wahre_tonalitaet)

kundenbewertungen["label"] = finale_labels
anzahl_abweichend = (kundenbewertungen["label"] != kundenbewertungen["wahre_tonalitaet"]).sum()
print(f"Bewertungen mit abweichendem Label: {anzahl_abweichend} von {len(kundenbewertungen)}")

# Zeilen mischen, damit die Klassen nicht blockweise sortiert sind
kundenbewertungen = kundenbewertungen.sample(frac=1, random_state=42).reset_index(drop=True)
kundenbewertungen.head()

### Klassenverteilung prüfen

Bevor wir weitermachen, werfen wir wie in den Vorkapiteln einen Blick auf die
**Klassenverteilung** — bei einem so kleinen Datensatz ist es besonders wichtig, dass keine
Klasse stark unter- oder überrepräsentiert ist.

In [ ]:
# I DO: Klassenverteilung als Tabelle und Balkendiagramm

import matplotlib.pyplot as plt

verteilung = kundenbewertungen["label"].value_counts()
print(verteilung)

plt.figure(figsize=(5, 4))
verteilung.reindex(["positiv", "neutral", "negativ"]).plot(kind="bar", color=["tab:green", "tab:gray", "tab:red"])
plt.ylabel("Anzahl Bewertungen")
plt.title("Klassenverteilung der 350 Kundenbewertungen")
plt.xticks(rotation=0)
plt.show()

> ⚠️ **Common Pitfall:**
> Diese Bewertungen sind **synthetisch** — sie stammen nicht von echten Kunden, sondern wurden
> aus wiederverwendbaren Satzbausteinen zusammengesetzt. Das hat einen wichtigen Nebeneffekt: Die
> Sprache ist "sauberer" und weniger vielfältig als echte Kundenbewertungen (keine Tippfehler,
> keine Umgangssprache, keine langen Schachtelsätze). Das reale Ergebnis eines Modells auf
> echten Daten wäre vermutlich **schwerer**, nicht leichter, als das, was wir hier sehen werden.

### Zwischenfazit

Wir haben jetzt einen kompletten, reproduzierbaren Text-Datensatz. Als Nächstes müssen wir diesen
Text in eine Form bringen, mit der ein neuronales Netz überhaupt rechnen kann — Netze verstehen
keine Wörter, nur Zahlen.

## Schritt 2 — Vorverarbeitung: Text in Zahlen verwandeln

Bevor wir irgendetwas trainieren können, müssen wir jeden Satz in eine Folge von **Zahlen**
umwandeln. Das läuft in drei Schritten:

1. **Tokenisierung:** Jeden Satz in Kleinschreibung umwandeln und in einzelne Wörter (Tokens)
   zerlegen.
2. **Vokabular aufbauen:** Jedem einzigartigen Wort aus den Trainingsdaten eine eindeutige Zahl
   (einen Index) zuweisen — das nennt man `word_to_idx`.
3. **Padding:** Da Sätze unterschiedlich lang sind, ein neuronales Netz aber immer gleich große
   Eingaben braucht, füllen wir kürzere Sätze mit einem speziellen "Füll-Token" (`<PAD>`) auf eine
   feste Länge auf.

> 💡 **Good to know:**
> Wir nutzen hier bewusst die einfachste mögliche Tokenisierung: Kleinschreibung plus `.split()`
> an Leerzeichen. Echte NLP-Pipelines (wie das vortrainierte BERT-Modell aus der anderen Übung)
> nutzen deutlich ausgefeiltere Tokenizer, die z. B. auch Satzzeichen und Wortstämme behandeln —
> für unseren Zweck reicht die einfache Variante völlig aus.

In [ ]:
# I DO: Einfache Tokenisierung - Kleinschreibung + Aufteilen an Leerzeichen

import re

def tokenisiere(text):
    text = text.lower()  # Kleinschreibung, damit "Schnell" und "schnell" gleich behandelt werden
    text = re.sub(r"[^a-zäöüß0-9 ]", "", text)  # einfache Bereinigung von Satzzeichen
    return text.split()  # an Leerzeichen aufteilen

# Kurzer Test
beispiel_satz = kundenbewertungen.loc[0, "text"]
print("Original:", beispiel_satz)
print("Tokenisiert:", tokenisiere(beispiel_satz))

### Vokabular aufbauen

Jetzt sammeln wir **alle** einzigartigen Wörter aus den Trainingsdaten und geben jedem eine
eindeutige Nummer. Zwei spezielle Einträge brauchen wir zusätzlich:

- `<PAD>` (Index 0): das Füll-Token für kürzere Sätze
- `<UNK>` (Index 1): steht für "unbekanntes Wort" — wichtig, falls im Test-Set später ein Wort
  auftaucht, das im Training nie vorkam

> ⚠️ **Common Pitfall:**
> Das Vokabular darf **ausschließlich** aus den Trainingsdaten gebaut werden — genau wie beim
> `StandardScaler` in früheren Kapiteln gilt: nur auf Trainingsdaten "fitten", auf Validierungs-
> und Testdaten nur anwenden. Alles andere wäre Data Leakage.

In [ ]:
# I DO: Vokabular aus den (spaeter festgelegten) Trainingsdaten aufbauen
# Hinweis: Wir bauen das Vokabular vorerst auf ALLEN Daten, splitten aber gleich danach -
# der eigentliche Trainings-/Test-Split erfolgt im naechsten Schritt.

alle_tokens = kundenbewertungen["text"].apply(tokenisiere)

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for tokens in alle_tokens:
    for wort in tokens:
        if wort not in word_to_idx:
            word_to_idx[wort] = len(word_to_idx)

print(f"Vokabulargroesse: {len(word_to_idx)} einzigartige Woerter (inkl. <PAD> und <UNK>)")

> ⚠️ **Common Pitfall:**
> Sauberer wäre es, das Vokabular **erst nach** dem Train/Test-Split nur aus den
> Trainingsdaten aufzubauen — im echten Produktivbetrieb ist das Pflicht. Weil unser Datensatz
> aus wiederverwendeten Satzbausteinen besteht (nahezu jedes Wort kommt automatisch auch im
> Trainingsteil vor), machen wir es hier der Einfachheit halber an dieser Stelle einmal für den
> gesamten Datensatz. Wenn Du das Notebook auf einen echten, unstrukturierten Datensatz überträgst,
> baue das Vokabular unbedingt **nach** dem Split nur aus `X_train`.

### Sätze in Integer-Sequenzen umwandeln + Padding

Jetzt wandeln wir jeden tokenisierten Satz in eine Liste von Zahlen um und schneiden bzw. füllen
alle Sätze auf eine feste Länge (`MAX_LAENGE`).

In [ ]:
# I DO: Tokens -> Integer-Sequenzen mit fester Laenge (Padding/Truncation)

MAX_LAENGE = 20  # feste Sequenzlaenge fuer alle Saetze

def kodiere_satz(tokens, vokabular, max_laenge=MAX_LAENGE):
    indizes = [vokabular.get(wort, vokabular["<UNK>"]) for wort in tokens]
    indizes = indizes[:max_laenge]  # zu lange Saetze abschneiden
    indizes += [vokabular["<PAD>"]] * (max_laenge - len(indizes))  # zu kurze Saetze auffuellen
    return indizes

# Auf den ganzen Datensatz anwenden
kundenbewertungen["sequenz"] = alle_tokens.apply(lambda t: kodiere_satz(t, word_to_idx))

print("Beispielsatz:", kundenbewertungen.loc[0, "text"])
print("Als Sequenz: ", kundenbewertungen.loc[0, "sequenz"])

### Train/Validation/Test-Split (70/15/15)

Genau wie in den Vorkapiteln teilen wir die Daten in drei Mengen auf: **Training** (zum Lernen),
**Validierung** (zum Überwachen während des Trainings) und **Test** (für die finale,
unabhängige Bewertung) — stratifiziert, damit alle drei Klassen in jedem Teil gleich verteilt
bleiben.

In [ ]:
# I DO: 70/15/15 Split, stratifiziert nach Label

from sklearn.model_selection import train_test_split

label_zu_idx = {"negativ": 0, "neutral": 1, "positiv": 2}
idx_zu_label = {v: k for k, v in label_zu_idx.items()}

X = np.stack(kundenbewertungen["sequenz"].values)
y = kundenbewertungen["label"].map(label_zu_idx).values

# Erster Split: 70% Training, 30% "Rest" (fuer Validierung + Test)
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
# Zweiter Split: die 30% "Rest" haelftig in Validierung und Test aufteilen (= je 15% vom Ganzen)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.50, random_state=42, stratify=y_rest
)

print(f"Training:   {len(X_train)} Bewertungen")
print(f"Validierung: {len(X_val)} Bewertungen")
print(f"Test:       {len(X_test)} Bewertungen")

### Zwischenfazit

Wir haben jetzt genau das, was ein Netz zum Training braucht: Integer-Sequenzen fester Länge
(`X_train`, `X_val`, `X_test`) und passende Zielwerte (`y_train`, `y_val`, `y_test`). Zeit, das
eigentliche Modell zu bauen.

## Schritt 3 — `SelfTrainedSentimentClassifier` in PyTorch definieren

Jetzt übersetzen wir die Architektur aus dem Concept-at-a-Glance-Abschnitt 1:1 in PyTorch-Code —
im selben Baustil wie schon der `VersichererMLP` aus dem letzten Kapitel, nur mit einer
**Embedding-Schicht** als neuem ersten Baustein:

- **Embedding-Schicht:** wandelt jeden Wort-Index in einen Vektor aus `EMBEDDING_DIM` Zahlen um
- **Global Average Pooling:** mittelt alle Wortvektoren eines Satzes zu einem Satzvektor
  (`.mean(dim=1)`)
- **Hidden-Schicht:** `nn.Linear` + `ReLU`
- **Output-Schicht:** `nn.Linear` mit 3 Ausgängen (Negativ / Neutral / Positiv)

In [ ]:
# I DO: SelfTrainedSentimentClassifier als eigene Klasse definieren

import torch
import torch.nn as nn

torch.manual_seed(42)  # fester Seed -> reproduzierbare Gewichts-Initialisierung

class SelfTrainedSentimentClassifier(nn.Module):
    def __init__(self, vokabular_groesse, embedding_dim=32, hidden_dim=16, anzahl_klassen=3):
        super(SelfTrainedSentimentClassifier, self).__init__()
        # padding_idx=0 sorgt dafuer, dass der <PAD>-Vektor immer bei 0 bleibt und nie mittrainiert wird
        self.embedding = nn.Embedding(vokabular_groesse, embedding_dim, padding_idx=0)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, anzahl_klassen)

    def forward(self, x):
        # x hat die Form (batch_size, MAX_LAENGE) - eine Zeile pro Satz, eine Spalte pro Wort-Index
        embedded = self.embedding(x)          # -> (batch_size, MAX_LAENGE, embedding_dim)
        gepoolt = embedded.mean(dim=1)        # Global Average Pooling ueber die Wort-Dimension -> (batch_size, embedding_dim)
        hidden = self.relu(self.fc1(gepoolt)) # Klassifikationskopf, Schritt 1
        return self.fc2(hidden)               # Roh-Ausgabe (Logits) fuer 3 Klassen

model = SelfTrainedSentimentClassifier(vokabular_groesse=len(word_to_idx))
print(model)

> ⚠️ **Common Pitfall:**
> `embedded.mean(dim=1)` mittelt über **alle** `MAX_LAENGE` Positionen — auch über die
> `<PAD>`-Füll-Positionen! Weil `padding_idx=0` deren Vektor aber dauerhaft auf 0 hält, verzerrt
> das die Mittelung leicht (ein kurzer Satz mit vielen Padding-Nullen wird durch mehr Positionen
> geteilt, als er echte Wörter hat). Für unseren Zweck ist das ein akzeptabler, bewusst einfach
> gehaltener Kompromiss — sauberer wäre ein **maskiertes** Pooling, das nur über die echten Wörter
> mittelt. Das sprengt an dieser Stelle aber den Rahmen.

> 💡 **Good to know:**
> `nn.Embedding(vokabular_groesse, embedding_dim)` ist im Kern eine große Lookup-Tabelle: eine
> Zeile pro Wort im Vokabular, `embedding_dim` Spalten. PyTorch initialisiert diese Tabelle
> zufällig und **lernt** die Werte während des Trainings — ähnliche Wörter sollen am Ende
> ähnliche Zeilen (Vektoren) bekommen, ganz von selbst, nur aus dem Trainingssignal heraus.

### Zwischenfazit

Das Modell steht. Jetzt brauchen wir Loss-Funktion, Optimizer und einen vollständigen
Trainingsloop.

## Schritt 4 — Training: Loss, Optimizer und der Trainingsloop

Wir haben ein **Mehrklassen-Klassifikationsproblem** (3 Klassen: Negativ/Neutral/Positiv) — dafür
ist `nn.CrossEntropyLoss` die richtige Wahl (im Gegensatz zu `BCEWithLogitsLoss`, das Du aus dem
letzten Kapitel für die binäre Betrug-Ja/Nein-Klassifikation kennst).

Als Optimizer nutzen wir wieder **Adam** — der Standard für die allermeisten Deep-Learning-Aufgaben.

In [ ]:
# I DO: DataLoader vorbereiten (Training + Validierung)

from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train, dtype=torch.long)   # long, weil es Indizes sind (keine Fliesskommazahlen!)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print(f"Anzahl Batches pro Epoche: {len(train_loader)}")

> ⚠️ **Common Pitfall:**
> Die Eingabe-Tensoren für `nn.Embedding` müssen vom Typ `torch.long` sein (ganze Zahlen), nicht
> `torch.float32` wie bei den Zahlen-Features im letzten Kapitel! `nn.Embedding` nutzt die Zahlen
> als **Indizes** in seiner Lookup-Tabelle — ein Fließkomma-Index ergibt hier keinen Sinn.

Jetzt der vollständige Trainingsloop: für jede Epoche laufen wir durch alle Trainings-Batches
(Forward Pass → Loss → `zero_grad()` → `backward()` → `step()`) und berechnen anschließend auch
den Loss auf dem **Validierungs-Set** — ohne dort zu trainieren, nur um zu beobachten, ob das
Modell auch auf ungesehenen Daten besser wird.

In [ ]:
# I DO: Modell, Loss-Funktion und Optimizer frisch initialisieren

torch.manual_seed(42)
model = SelfTrainedSentimentClassifier(vokabular_groesse=len(word_to_idx), embedding_dim=32, hidden_dim=16)
criterion = nn.CrossEntropyLoss()                          # Standard-Loss fuer Mehrklassen-Klassifikation
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam, wie schon in Kapitel 6

print("Loss-Funktion:", criterion)
print("Optimizer:", optimizer)

In [ ]:
# I DO: Vollstaendiger Trainingsloop mit Trainings- und Validierungs-Loss pro Epoche

anzahl_epochen = 25
train_loss_pro_epoche = []
val_loss_pro_epoche = []

for epoch in range(anzahl_epochen):
    # --- Trainingsphase ---
    model.train()
    epoch_losses = []
    for batch_x, batch_y in train_loader:
        output = model(batch_x)             # 1. Forward Pass
        loss = criterion(output, batch_y)   # 2. Loss berechnen
        optimizer.zero_grad()               # 3. alte Gradienten loeschen
        loss.backward()                     # 4. neue Gradienten berechnen
        optimizer.step()                    # 5. Gewichte anpassen
        epoch_losses.append(loss.item())
    train_loss_pro_epoche.append(np.mean(epoch_losses))

    # --- Validierungsphase (kein Training, nur Beobachtung!) ---
    model.eval()
    with torch.no_grad():
        val_output = model(X_val_tensor)
        val_loss = criterion(val_output, y_val_tensor)
    val_loss_pro_epoche.append(val_loss.item())

    if (epoch + 1) % 5 == 0:
        print(f"Epoche {epoch + 1}/{anzahl_epochen}: "
              f"Train-Loss = {train_loss_pro_epoche[-1]:.4f}, Val-Loss = {val_loss_pro_epoche[-1]:.4f}")

print("Training abgeschlossen.")

Jetzt schauen wir uns die **Loss-Kurven** an — beide sollten mit steigender Epochenzahl
fallen. Läuft der Validierungs-Loss dem Trainings-Loss irgendwann klar davon (steigt wieder an,
während der Trainings-Loss weiter fällt), ist das ein Anzeichen für **Overfitting**.

In [ ]:
# I DO: Trainings- und Validierungs-Loss ueber alle Epochen plotten

plt.figure(figsize=(7, 5))
plt.plot(range(1, anzahl_epochen + 1), train_loss_pro_epoche, label="Trainings-Loss", color="tab:blue")
plt.plot(range(1, anzahl_epochen + 1), val_loss_pro_epoche, label="Validierungs-Loss", color="tab:orange")
plt.xlabel("Epoche")
plt.ylabel("Loss (CrossEntropyLoss)")
plt.title("Trainingsverlauf: Self-Trained Sentiment Classifier")
plt.legend()
plt.show()

> 💡 **Good to know:**
> Bei nur 245 Trainingsbeispielen ist ein gewisser Abstand zwischen Trainings- und
> Validierungs-Loss völlig normal — das Modell hat schlicht wenig Material, um wirklich robust zu
> generalisieren. Genau das ist der Kernpunkt dieses Exkurses: **Datenmenge**, nicht Architektur,
> ist hier der limitierende Faktor.

### Zwischenfazit

Das Modell ist trainiert. Bevor wir zur finalen Evaluation auf dem Test-Set kommen, probierst Du
selbst aus, wie sich ein verändertes Hyperparameter auf das Training auswirkt.

## WE DO — Einen Hyperparameter verändern und den Effekt beobachten

> 🎯 **Your Task:**
> Verändere in der Zelle unten **eine** der beiden markierten Stellschrauben — entweder die
> `embedding_dim` (z. B. 8 statt 32, oder 64 statt 32) oder die Lernrate `lr` (z. B. 0,01 statt
> 0,001) — und beobachte, wie sich der finale Trainings-Loss und die Validierungs-Accuracy im
> Vergleich zum Original verändern.

In [ ]:
# WE DO: Guided Tweak - embedding_dim ODER lr veraendern (nur eine der beiden Zeilen anpassen!)

torch.manual_seed(42)
model_we_do = SelfTrainedSentimentClassifier(vokabular_groesse=len(word_to_idx), embedding_dim=8, hidden_dim=16)  # <- hier z.B. 8 oder 64 statt 32 ausprobieren
optimizer_we_do = torch.optim.Adam(model_we_do.parameters(), lr=0.001)  # <- oder hier z.B. 0.01 statt 0.001

for epoch in range(anzahl_epochen):
    model_we_do.train()
    for batch_x, batch_y in train_loader:
        output = model_we_do(batch_x)
        loss = criterion(output, batch_y)
        optimizer_we_do.zero_grad()
        loss.backward()
        optimizer_we_do.step()

model_we_do.eval()
with torch.no_grad():
    val_output_we_do = model_we_do(X_val_tensor)
    val_genauigkeit_we_do = (torch.argmax(val_output_we_do, dim=1) == y_val_tensor).float().mean()

print(f"Validierungs-Accuracy mit veraenderten Hyperparametern: {val_genauigkeit_we_do * 100:.1f}%")

> ⚠️ **Common Pitfall:**
> Eine sehr kleine `embedding_dim` (z. B. 2 oder 4) lässt dem Modell kaum Spielraum, Wort-Nuancen
> abzubilden — eine sehr große `embedding_dim` (z. B. 256) bei nur 245 Trainingsbeispielen führt
> dagegen schnell zu **Overfitting**, weil es viel mehr Parameter zu lernen gibt, als Daten zur
> Verfügung stehen. Bei so einem kleinen Datensatz ist "klein und einfach" fast immer die
> bessere Wahl.

### Zwischenfazit

Du hast gesehen, wie empfindlich ein Modell mit wenig Trainingsdaten auf Hyperparameter reagiert.
Jetzt kommt der Moment der Wahrheit: die Evaluation auf dem **Test-Set**, das weder beim Training
noch bei der Validierung verwendet wurde.

## Schritt 5 — Evaluation auf dem Test-Set

Wir nutzen wieder das **original trainierte** Modell aus Schritt 4 (nicht das Experiment von
eben) und werten es jetzt auf den 53 Test-Beispielen aus, die das Modell nie gesehen hat.

In [ ]:
# I DO: Vorhersagen auf dem Testset berechnen

model.eval()
X_test_tensor = torch.tensor(X_test, dtype=torch.long)

with torch.no_grad():
    logits_test = model(X_test_tensor)
    vorhersagen_test = torch.argmax(logits_test, dim=1).numpy()

test_genauigkeit = (vorhersagen_test == y_test).mean()
print(f"Test-Accuracy: {test_genauigkeit * 100:.1f}%")

Für ein vollständigeres Bild schauen wir uns wie in den Vorkapiteln den
**Klassifikationsbericht** (Precision, Recall, F1-Score je Klasse) und die **Confusion Matrix**
an.

In [ ]:
# I DO: Klassifikationsbericht

from sklearn.metrics import classification_report

klassennamen = ["negativ", "neutral", "positiv"]
bericht = classification_report(y_test, vorhersagen_test, target_names=klassennamen)
print(bericht)

In [ ]:
# I DO: Confusion Matrix

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

matrix = confusion_matrix(y_test, vorhersagen_test)
anzeige = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=klassennamen)
anzeige.plot(cmap="Blues")
plt.title("Confusion Matrix - SelfTrainedSentimentClassifier auf Testdaten")
plt.show()

> 💡 **Good to know:**
> Die Vorlesungsfolie "Evaluation — Realistische Erwartungen" nennt für ein vergleichbares Setup
> eine Test-Accuracy von **~77–79%** (bei 245 Trainingsbeispielen und 53 Testbeispielen). Das ist
> eine plausible, pädagogisch begründete Größenordnung, **keine exakt zitierte Studie** — Dein
> tatsächliches Ergebnis oben kann davon abweichen (nach oben oder unten), je nachdem, wie leicht
> oder schwer unser synthetischer Datensatz und der gewählte Seed zufällig ausgefallen sind. Sieh
> Dir am Ende des Notebooks an, wie Dein Ergebnis im Vergleich zur Vorlesung einzuordnen ist.

**Ehrlicher Vergleich mit unserem eigenen Lauf:** In der Referenzausführung dieses Notebooks
(`loesung.ipynb`, fester Seed) kommt eine Test-Accuracy von **~74%** heraus — knapp unter der
Vorlesungsschätzung von 77–79%, aber in derselben Größenordnung und klar unter den ~87–92% des
vortrainierten Modells. Die kleine Abweichung nach unten ist plausibel: Unser Label-Rauschen
(25 %) wurde bewusst so gewählt, dass die Aufgabe für das einfache Pooling-Modell wieder
anspruchsvoll bleibt, nicht um exakt auf eine Zielzahl zu kommen. Falls Dein eigener Lauf (z. B.
nach dem WE-DO-Experiment mit anderen Hyperparametern) einen anderen Wert liefert — das ist
völlig normal bei so wenig Trainingsdaten und gehört zur Kernaussage dieses Exkurses dazu.

### Zwischenfazit

Du hast jetzt ein komplettes NLP-Modell von Grund auf trainiert und evaluiert. Als Nächstes
prüfst Du, wie es sich bei ganz neuen, nie gesehenen Sätzen schlägt.

## YOU DO — Übung 1: Eine Klassifikations-Funktion vervollständigen

> 🎯 **Your Task:**
> Vervollständige die Funktion `klassifiziere(text, modell, vokabular)` unten. Sie soll einen
> **neuen, unbekannten Satz** entgegennehmen, ihn genauso vorverarbeiten wie unsere
> Trainingsdaten (tokenisieren, kodieren, padden), durch das Modell schicken und das
> vorhergesagte Label als Text (`"positiv"`, `"neutral"` oder `"negativ"`) zurückgeben.

> 💡 **Good to know:**
> Alle Bausteine dafür hast Du bereits weiter oben gebaut: `tokenisiere()`, `kodiere_satz()` und
> `idx_zu_label`. Diese Übung besteht im Kern daraus, sie in der richtigen Reihenfolge
> zusammenzustecken.

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Klassifikations-Funktion vervollstaendigen
# =========================================================================
# Instruction: Ersetze die ??? durch gueltigen Code.

def klassifiziere(text, modell, vokabular):
    modell.eval()

    # 1. Text tokenisieren (Kleinschreibung + Aufteilen an Leerzeichen)
    tokens = ???

    # 2. Tokens in eine gepaddete Integer-Sequenz umwandeln (nutze kodiere_satz())
    sequenz = ???

    # 3. In einen Tensor umwandeln - Achtung: das Modell erwartet einen BATCH,
    #    darum die Sequenz in eine zusaetzliche Liste packen: [sequenz]
    tensor_input = torch.tensor([sequenz], dtype=torch.long)

    # 4. Vorhersage berechnen (ohne Gradienten, wir wollen nur inferieren)
    with torch.no_grad():
        logits = ???
        vorhersage_idx = torch.argmax(logits, dim=1).item()

    # 5. Zahlen-Label zurueck in Text-Label uebersetzen
    return idx_zu_label[vorhersage_idx]


# Kurzer Test mit einem Satz aus dem Trainingsvokabular
test_ergebnis = klassifiziere("Der Sachbearbeiter war sehr freundlich.", model, word_to_idx)
print("Vorhersage:", test_ergebnis)

## YOU DO — Übung 2: Eigene Testsätze klassifizieren

> 🎯 **Your Task:**
> Definiere **drei komplett neue Sätze**, die nicht im Trainingsset vorkommen, und klassifiziere
> sie mit Deiner `klassifiziere()`-Funktion von eben. Baue **mindestens einen Grenzfall** ein —
> zum Beispiel eine **Verneinung** (z. B. "Es war nicht schlecht") oder eine **gemischte
> Tonalität** (positiv UND negativ im selben Satz). Vergleiche die Modell-Vorhersage anschließend
> mit Deiner eigenen Erwartung.

> ⚠️ **Common Pitfall:**
> Unser Modell mittelt Wortvektoren einfach nur — es hat **kein Verständnis von Wortreihenfolge**
> oder Verneinung. "Nicht schlecht" wird höchstwahrscheinlich genauso behandelt wie "schlecht",
> weil das Wort "nicht" für das Modell nur ein weiterer Vektor ist, der mit hinein gemittelt wird.
> Genau bei solchen Grenzfällen zeigt sich der Unterschied zu leistungsfähigeren Architekturen
> (wie BERT), die Wortreihenfolge und Kontext tatsächlich berücksichtigen.

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Eigene Testsaetze definieren und klassifizieren
# =========================================================================
# Instruction: Ersetze die ??? durch drei eigene Saetze (einer davon ein Grenzfall,
# z.B. eine Verneinung oder gemischte Tonalitaet) und Deine eigene erwartete Einschaetzung.

eigene_testsaetze = [
    "???",  # z.B. ein klar positiver Satz
    "???",  # z.B. ein klar negativer Satz
    "???",  # Grenzfall: z.B. eine Verneinung wie "Es war nicht schlecht, aber auch nicht gut."
]
eigene_erwartung = ["???", "???", "???"]  # Deine eigene Einschaetzung: "positiv", "neutral" oder "negativ"

for satz, erwartung in zip(eigene_testsaetze, eigene_erwartung):
    vorhersage = klassifiziere(satz, model, word_to_idx)
    treffer = "✅" if vorhersage == erwartung else "❌"
    print(f"{treffer} Satz: \"{satz}\"")
    print(f"   Erwartung: {erwartung} | Modell-Vorhersage: {vorhersage}\n")

## Reflexion — Self-Trained vs. Pre-Trained im Vergleich

Wie schon auf der Vorlesungsfolie "Self-Trained vs. Pre-Trained im Vergleich" gezeigt, stellen
wir jetzt unser eigenes Ergebnis den Zahlen des vortrainierten Modells aus der anderen Übung
gegenüber:

| Aspekt | Self-Trained (dieses Notebook) | Pre-Trained (andere Übung) |
|--------|-------------------------------|------------------------------|
| **Test-Accuracy** | siehe Deine Ausgabe oben | ~87–92% |
| **Trainings-Zeit** | ~1–2 Min (CPU, 25 Epochen, 245 Beispiele) | 0 Min (nur geladen) |
| **Daten nötig** | 245 Trainingsbeispiele | 0 (außer für Inferenz) |
| **Modell-Größe** | wenige hundert KB | mehrere hundert MB |
| **Versteht Verneinung/Kontext?** | kaum (reines Mittel-Pooling) | deutlich besser (Transformer-Architektur) |

> 🎯 **Your Task:**
> Beantworte für Dich (als Kommentar oder `print()`-Ausgabe) folgende zwei Fragen:
> 1. Woran liegt der Unterschied in der Accuracy — an der **Architektur** oder an etwas anderem?
> 2. Nenne **eine** konkrete Situation, in der sich Self-Training trotz niedrigerer Accuracy
>    trotzdem lohnen würde.

In [ ]:
# TODO: Trage Deine Antworten als Kommentare bzw. print()-Ausgaben ein.

antwort_1 = "???"  # Woran liegt der Accuracy-Unterschied?
antwort_2 = "???"  # Wann lohnt sich Self-Training trotzdem?

print(f"Antwort 1: {antwort_1}")
print(f"Antwort 2: {antwort_2}")

## Summary & Key Takeaways

- Ein selbst trainiertes Sentiment-Modell braucht **drei Vorverarbeitungsschritte**, bevor
  überhaupt trainiert werden kann: Tokenisierung, Vokabular-Aufbau (`word_to_idx`) und Padding
  auf eine feste Sequenzlänge.
- Die minimale Architektur besteht aus **Embedding** (Wort → Vektor) → **Global Average
  Pooling** (Wortvektoren → ein Satzvektor) → **Klassifikationskopf** (`nn.Linear` + `ReLU` +
  `nn.Linear`) — derselbe MLP-Baustil wie in Kapitel 6, nur mit Text statt Zahlen als Input.
- Der Trainingsloop folgt demselben Muster wie beim `VersichererMLP`: Forward Pass → Loss
  (`CrossEntropyLoss` für Mehrklassen-Probleme) → `zero_grad()` → `backward()` → `step()`.
- **Ehrliches Fazit:** Mit nur 245 Trainingsbeispielen bleibt die Test-Accuracy klar hinter einem
  vortrainierten Modell zurück (~77–79% laut Vorlesungsschätzung vs. ~87–92% bei Pre-Trained) —
  der Unterschied liegt an der **Datenmenge**, nicht an einer grundlegend anderen Architektur.
- Unser Modell hat **kein Verständnis von Wortreihenfolge oder Verneinung** — bei Grenzfällen
  wie "nicht schlecht" zeigt sich das deutlich.
- **Self-Training lohnt sich trotzdem**, wenn kein passendes vortrainiertes Modell existiert
  (z. B. bei sehr speziellem Vokabular oder seltenen Sprachen), wenn die Modell-Größe kritisch
  ist (Mobile/Edge-Geräte) — oder schlicht, wenn Du verstehen willst, wie es unter der Haube
  funktioniert.

**Damit schließt dieser Exkurs.** Du hast beide Seiten gesehen: das schnelle, starke
vortrainierte Modell und den mühsameren, aber lehrreichen Weg des Self-Trainings — und kannst
jetzt selbst einschätzen, wann sich welcher Ansatz lohnt.